# 04 — Spatial transcriptomics

**CIAD single-cell workshop**

Adapted from the Seurat *Analysis of spatial datasets* vignette:
<https://satijalab.org/seurat/articles/spatial_vignette>

Everything so far threw away one thing: **where the cell was**. Tissue was
dissociated into a suspension, and position was lost before sequencing began.

Spatial transcriptomics keeps it. You measure expression *and* know where in the
tissue each measurement came from.

**The data.** A 10x Visium slide of a mouse brain, sagittal section, anterior
half. The tissue is placed on a slide printed with ~5,000 spots in a regular
grid, each 55 µm across. Every spot captures the RNA above it, tagged with a
barcode that identifies its position. A photograph of the stained tissue is taken
first, so expression can be laid back over the anatomy.

**The catch, and it matters.** A 55 µm spot is larger than a cell. Each spot
holds roughly 1–10 cells, and what you measure is their mixture. So a "spot" is
not a cell, and this is not single-cell data — it is spatially resolved bulk, at
very fine grain. Everything else follows from that.

**What we do**

1. load the data, and see what an image inside a Seurat object looks like
2. quality control, which behaves differently here
3. normalise with `SCTransform`
4. plot known genes onto the anatomy
5. cluster with **no spatial information at all**, then plot the clusters onto the
   tissue and see what happened

About 30 minutes.

## Setup

This notebook uses its own setup file. It adds `glmGamPoi`, which makes
`SCTransform` much faster — the other notebooks do not use `SCTransform`, so they
do not install it.

In [ ]:
source("https://raw.githubusercontent.com/MartinLoza/CIAD_workshop_sc/main/setup/setup_spatial.R")

## 1. Loading a Visium dataset

Two files, and they are different in kind:

- **`filtered_feature_bc_matrix.h5`** — the counts. Spots x genes, in HDF5, which
  is why our setup installs `hdf5r`.
- **`spatial.tar.gz`** — the photograph of the tissue, plus a table giving the
  pixel coordinates of every spot on that photograph.

`Load10X_Spatial()` expects them arranged in one directory, with the image files
in a `spatial/` subfolder — the layout Space Ranger produces. We recreate it.

In [ ]:
base <- "https://cf.10xgenomics.com/samples/spatial-exp/1.1.0/V1_Mouse_Brain_Sagittal_Anterior"
h5   <- "V1_Mouse_Brain_Sagittal_Anterior_filtered_feature_bc_matrix.h5"

dir.create("visium", showWarnings = FALSE)

download.file(file.path(base, h5), file.path("visium", h5), quiet = TRUE)
download.file(file.path(base, "V1_Mouse_Brain_Sagittal_Anterior_spatial.tar.gz"),
              "spatial.tar.gz", quiet = TRUE)

untar("spatial.tar.gz", exdir = "visium")

list.files("visium", recursive = TRUE)

Look at what came out of the tarball:

- `tissue_lowres_image.png` — the photograph, downsampled
- `tissue_positions_list.csv` — one row per spot: its barcode, whether it sits on
  tissue, and its row/column and pixel coordinates
- `scalefactors_json.json` — how to convert between full-resolution pixels and the
  downsampled image

That is the whole spatial part. The rest is a counts matrix like any other.

In [ ]:
brain <- Load10X_Spatial(data.dir = "visium", filename = h5)

brain

### What is different about this object

Compare that summary to notebook 01's. Two things changed.

The assay is called **`Spatial`**, not `RNA` — a naming convention, nothing deeper.

And the object now carries an **image**. That is new: `Images()` lists them, and
the coordinates of every spot come with it.

In [ ]:
cat("assay      :", DefaultAssay(brain), "\n")
cat("spots      :", ncol(brain), "\n")
cat("genes      :", nrow(brain), "\n")
cat("images     :", Images(brain), "\n\n")

coords <- GetTissueCoordinates(brain)
cat("coordinates, first 5 spots:\n")
head(coords, 5)

Two numbers per spot, and they are pixel positions on the photograph. That is
the entire difference between this and the data in notebook 01.

Spot names are barcodes, exactly as before.

In [ ]:
head(colnames(brain), 3)

head(brain@meta.data, 3)

## 2. Quality control, and why it differs here

`nCount_Spatial` is total counts per spot. In notebook 02 a low-count droplet was
a dying cell or an empty droplet, and we removed it.

Here, low counts may simply mean **fewer cells under that spot**. A region of
sparse tissue gives low counts because that is what the anatomy is like, not
because anything went wrong.

So look at the counts *in space* before deciding anything.

In [ ]:
options(repr.plot.width = 14, repr.plot.height = 6)

p1 <- VlnPlot(brain, features = "nCount_Spatial", pt.size = 0.1) + NoLegend()
p2 <- SpatialFeaturePlot(brain, features = "nCount_Spatial") +
  theme(legend.position = "right")

p1 + p2

The right panel is the whole argument. Counts vary smoothly across the
section and the pattern follows anatomy — dense structures give high counts,
ventricles and fibre tracts give low ones.

Filter that on a global threshold and you delete anatomy, not bad data.

### ✏️ Exercise 1

Where on the section are the counts lowest, and is that a technical problem or a
biological one?

Nothing to code. Look at the right panel and say what you would do.

*Your answer:*



## 3. Normalisation

The variance in this data depends on how many cells sat under each spot, which
varies systematically across the tissue. Plain log-normalisation assumes that
variation is technical and removes it — here, some of it is anatomy.

`SCTransform` models each gene's counts with a regularised negative binomial and
normalises without that assumption. It is the vignette's recommendation for
spatial data, and the reason this notebook installs `glmGamPoi`.

A minute or so.

In [ ]:
brain <- SCTransform(brain, assay = "Spatial", verbose = FALSE)

Assays(brain)

A new assay, `SCT`, now the default. `Spatial` still holds the raw counts.

## 4. Genes on anatomy

Now the part that makes spatial data worth the trouble.

`SpatialFeaturePlot()` puts expression back on the photograph. Two genes with
well-known, very localised expression in the mouse brain:

- **`Hpca`** — hippocalcin, essentially restricted to the hippocampus
- **`Ttr`** — transthyretin, made by the choroid plexus, a small structure in the
  ventricles

In [ ]:
options(repr.plot.width = 14, repr.plot.height = 6)

SpatialFeaturePlot(brain, features = c("Hpca", "Ttr"))

No clustering, no annotation, no reference. Two genes plotted on a
photograph, and the hippocampus and choroid plexus draw themselves.

### Making the plots readable

Two arguments do most of the work. `pt.size.factor` scales the spots — larger
fills the tissue, smaller shows the image beneath. `alpha` makes low-expressing
spots transparent so the pattern stands out.

In [ ]:
options(repr.plot.width = 14, repr.plot.height = 6)

p1 <- SpatialFeaturePlot(brain, features = "Ttr", pt.size.factor = 1) +
  ggtitle("small spots")
p2 <- SpatialFeaturePlot(brain, features = "Ttr", alpha = c(0.1, 1)) +
  ggtitle("faded where low")

p1 + p2

### ✏️ Exercise 2

Plot two more genes with strong regional expression in the mouse brain.
Suggestions: `Mbp` (myelin, so white matter), `Calb1` (calbindin), `Pcp4`
(Purkinje cell protein 4), `Nrgn` (cortex).

Fill in the blank:

In [ ]:
# SpatialFeaturePlot(brain, features = c(______, ______))

## 5. Clustering

Here is the part worth pausing on.

We now run **exactly the pipeline from notebook 02** — PCA, neighbours, clusters,
UMAP. Nothing in it knows about position. The spatial coordinates are not passed
to any of these functions. As far as the clustering is concerned, these are 2,700
unrelated samples.

In [ ]:
brain <- RunPCA(brain, assay = "SCT", verbose = FALSE)
brain <- FindNeighbors(brain, dims = 1:30, verbose = FALSE)
brain <- FindClusters(brain, resolution = 0.8, verbose = FALSE)
brain <- RunUMAP(brain, dims = 1:30, verbose = FALSE)

table(Idents(brain))

Now plot those clusters twice: on the UMAP, and on the tissue.

In [ ]:
options(repr.plot.width = 16, repr.plot.height = 7)

p1 <- DimPlot(brain, reduction = "umap", label = TRUE) + NoLegend() +
  ggtitle("UMAP — no spatial information used")
p2 <- SpatialDimPlot(brain, label = TRUE, label.size = 3) +
  ggtitle("the same clusters, on the tissue")

p1 + p2

**That is the result of this notebook.**

The clusters were computed from expression alone. Position was never given to the
algorithm. Yet plotted back onto the section they reconstruct brain anatomy —
cortical layers as stacked bands, the hippocampus as a distinct curve, white
matter tracts separated from grey.

Which is a strong statement about the data: cells in a tissue region are alike
enough in expression that clustering finds the region without being told it
exists.

It is also a check on your analysis. If clusters landed as speckle scattered
across the section, something would be wrong — either the clustering or the
tissue.

### One cluster at a time

Overlapping colours are hard to read. `SpatialDimPlot()` can highlight
individual clusters instead.

In [ ]:
options(repr.plot.width = 16, repr.plot.height = 8)

SpatialDimPlot(
  brain,
  cells.highlight = CellsByIdentities(brain, idents = c(1, 2, 3, 5)),
  facet.highlight = TRUE,
  ncol = 4
)

### ✏️ Exercise 3

Pick a cluster that forms a compact region and find what marks it.

Fill in the blanks:

In [ ]:
# my_cluster <- ______
#
# markers <- FindMarkers(brain, ident.1 = my_cluster, only.pos = TRUE, verbose = FALSE)
# head(markers, 5)
#
# SpatialFeaturePlot(brain, features = rownames(markers)[1])

## 6. Markers of a spatial region

The same `FindMarkers()` as notebook 02. What changed is that the result can be
plotted onto anatomy, so a marker is checkable by eye.

In [ ]:
cluster_markers <- FindMarkers(brain, ident.1 = 1, only.pos = TRUE, verbose = FALSE)

head(cluster_markers, 5)

In [ ]:
options(repr.plot.width = 14, repr.plot.height = 6)

SpatialFeaturePlot(brain, features = rownames(cluster_markers)[1:2])

## What we did, and what we left out

From a slide to labelled anatomy:

- loaded counts plus an image into one object
- saw that QC has to account for tissue structure, not just technical failure
- normalised with `SCTransform`
- plotted genes onto the photograph and recognised structures from two genes
- clustered without spatial information, and recovered anatomy anyway

**Deliberately left out**, all of it in the vignette if you want it:

- **Multiple slices.** Sections are integrated much like the batches in notebook
  03 — the machinery you already have.
- **Label transfer from single-cell data.** Because a spot is several cells, a
  natural next step is to estimate which cell types compose each spot, using an
  annotated scRNA-seq reference and `FindTransferAnchors()`. It needs a large
  reference dataset and considerable time.
- **Spatially variable genes.** `FindSpatiallyVariableFeatures()` finds genes with
  spatial pattern without using clusters, via Moran's I. Conceptually neat, slow
  in practice.
- **Imaging-based platforms** — Xenium, MERFISH, CosMx — which are genuinely
  single-cell and come with cell boundaries. Covered in a
  [separate vignette](https://satijalab.org/seurat/articles/seurat5_spatial_vignette_2).

The thing to carry away: spatial data is analysed with the tools you already
learned. What is new is that every result can be put back on the tissue and
checked against anatomy — which is a much stronger form of validation than a UMAP.

---

### Answers

<details>
<summary>Click to expand</summary>

**Exercise 1**

Counts are lowest over fibre tracts and ventricles. That is biological: fewer
cells, and cells with less RNA, sit under those spots. Removing low-count spots
would delete those structures from the section.

For spatial data, QC is better aimed at *technical* failure — spots off the
tissue, or a damaged edge of the section — which you find by looking at the
image, not by thresholding counts. `Load10X_Spatial()` has already dropped
off-tissue spots for us.

**Exercise 2**

```r
SpatialFeaturePlot(brain, features = c("Mbp", "Nrgn"))
```

`Mbp` marks white matter and `Nrgn` the cortex, so the two are close to
complementary.

**Exercise 3**

```r
my_cluster <- 3
markers <- FindMarkers(brain, ident.1 = my_cluster, only.pos = TRUE, verbose = FALSE)
head(markers, 5)
SpatialFeaturePlot(brain, features = rownames(markers)[1])
```

If the top marker's expression pattern matches the cluster's position on the
section, the cluster is a real region. If it does not, be suspicious.

</details>